# Mask 단위 확정 pilot: lead × time patch

이 노트북은 기존 전처리 파일을 수정하지 않습니다. Train과 validation만 읽고 test는 읽지 않습니다.
먼저 시각화와 단위 검사를 하고, 원할 때만 CPU smoke 또는 GPU pilot을 실행합니다.

**중요:** smoke는 코드 검증용이며 최적 patch/mask ratio를 결정하는 성능 실험이 아닙니다.
GPU pilot도 1-seed 단기 탐색입니다. 상위 두 조건은 같은 모델과 더 긴 학습, 3개 seed로 재검증하세요.
예측 가능성 estimator와 Easy/Medium/Hard sampler는 아직 포함하지 않습니다.


## 1. 환경 준비
기존 preprocessing 커널을 사용하세요. 시각화에는 numpy/pandas/matplotlib만 필요합니다.
학습에는 torch와 scikit-learn도 필요합니다.

CPU 환경에서 필요한 경우 아래 설치 셀의 주석을 해제하세요. GPU 서버에서는 CPU 전용 torch를
설치하지 말고 서버 관리자 안내 및 [PyTorch 설치 안내](https://pytorch.org/get-started/locally/)를 따르세요.
Seraph의 계정/스케줄러/할당 정책은 확인되지 않았으므로 임의의 접속·작업 제출 명령은 제공하지 않습니다.


In [ ]:
# 필요한 경우에만 실행하고 커널을 재시작하세요.
# %pip install numpy pandas matplotlib scikit-learn
# CPU에서만:
# %pip install torch --index-url https://download.pytorch.org/whl/cpu


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import importlib.metadata
import importlib.util
import json
import math
import os
import random
import time
import uuid
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

try:
    import torch
    from torch import nn
    from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
    TRAINING_AVAILABLE = True
except ImportError:
    TRAINING_AVAILABLE = False
    torch = None

print("학습 라이브러리:", TRAINING_AVAILABLE)
if TRAINING_AVAILABLE:
    print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())


## 2. 실행 설정과 경로

기본 RUN_TRAINING=False: CPU에서 부담 없는 입력/patch/mask 검증과 시각화만 실행합니다.
MODE="smoke", RUN_TRAINING=True: 작은 모델, 64 ECG, 8 update로 한 조합만 동작 확인합니다.
MODE="pilot", RUN_TRAINING=True: GPU가 필요하며 9개 조건을 각각 1,000 update 학습합니다.

MODEL_SIZE="baseline"의 GPU pilot은 encoder 6층/192차원/6 heads, decoder 2층/128차원/4 heads입니다.
MODEL_SIZE="stmem-small"은 encoder 12층/384차원/6 heads, decoder 4층/256차원/4 heads로 키웁니다.
기본 MAE의 global mask, learned lead/time embedding, global decoder, loss, optimizer와 pooling을 유지합니다.
크기만의 효과를 비교하려면 두 크기에 같은 INITIALIZATION과 학습·평가 설정을 사용하세요.
기본 CPU smoke 모델은 훨씬 작습니다. 큰 모델도 smoke 데이터/업데이트로 코드 검사할 수 있지만
smoke 결과를 최종 설정 선택에 쓰지 마세요.

서버에서는 전처리 artifacts/ptbxl_250hz 폴더 전체를 옮긴 뒤 ARTIFACT_DIR_OVERRIDE를 지정하세요.
원본 WFDB를 다시 가져올 필요는 없습니다. 실제 manifest의 필터 설정을 기록하며
노션의 50 Hz 계획과 현재 40 Hz 산출물을 혼동하지 않습니다.


In [ ]:
MODE = "smoke"                 # "smoke" 또는 "pilot"
RUN_TRAINING = False            # True로 바꿔야 학습 시작
SEED = 42
MODEL_SIZE = "baseline"         # baseline 또는 stmem-small (ST-MEM small의 폭/깊이)
INITIALIZATION = "independent"  # legacy: reproduce old cloned-layer initialization
SUBSET_SEED = None            # None: use SEED; fix across model seeds for paired comparisons
EVAL_MASK_SEED = None         # None: use SEED; fix validation masks across model seeds
DATA_SEED = SEED if SUBSET_SEED is None else SUBSET_SEED
EVAL_SEED = SEED if EVAL_MASK_SEED is None else EVAL_MASK_SEED
PATCH_LENGTHS = [50, 125, 250]   # 250 Hz: 0.2 / 0.5 / 1.0초
MASK_RATIOS = [0.50, 0.75, 0.90]
ARTIFACT_DIR_OVERRIDE = None    # 예: "/your/path/ptbxl_250hz"
OUTPUT_ROOT_OVERRIDE = None    # None: 이 Git 저장소의 experiments/artifacts 아래
ALLOW_CPU_PILOT = False         # 안전장치: GPU pilot의 CPU 실행 차단
STEPS_OVERRIDE = None          # 예: 3000. None이면 기존 smoke/pilot update 수 유지
DIAGNOSTICS = dict(enabled=True, eval_every=200, eval_n=32, repeats=1,
                   grad_every=50, examples=3, random_encoder_probe=True,
                   cross_lead_fit_n=256, cross_lead_ridge=1e-3)

cwd = Path.cwd().resolve()
REPO_DIR = next((p for p in [cwd, *cwd.parents]
                 if p.name == "predictability-guided-ecg-masking"), None)
if REPO_DIR is None:
    REPO_DIR = next((p / "predictability-guided-ecg-masking"
                     for p in [cwd, *cwd.parents]
                     if (p / "predictability-guided-ecg-masking" / "preprocessing").is_dir()), None)
if REPO_DIR is None and ARTIFACT_DIR_OVERRIDE is None:
    raise FileNotFoundError("저장소 안에서 실행하거나 ARTIFACT_DIR_OVERRIDE를 설정하세요.")
REPO_DIR = REPO_DIR or cwd
ARTIFACT_DIR = (Path(ARTIFACT_DIR_OVERRIDE).expanduser().resolve()
                if ARTIFACT_DIR_OVERRIDE else REPO_DIR / "preprocessing/artifacts/ptbxl_250hz")
OUTPUT_FAMILY = "mae_stmem_size_pilot" if MODEL_SIZE == "stmem-small" else "masking_pilot"
OUTPUT_ROOT = (Path(OUTPUT_ROOT_OVERRIDE).expanduser().resolve()
               if OUTPUT_ROOT_OVERRIDE else REPO_DIR / "experiments/artifacts" / OUTPUT_FAMILY)
assert MODE in {"smoke", "pilot"}

PROFILE = {
    "smoke": dict(train_n=64, val_n=32, batch=4, steps=8, eval_n=16,
                  probe_steps=8, probe_batch=16, enc_dim=32, enc_depth=1,
                  enc_heads=4, dec_dim=32, dec_depth=1, dec_heads=4),
    "pilot": dict(train_n=5000, val_n=None, batch=32, steps=1000, eval_n=128,
                  probe_steps=300, probe_batch=256, enc_dim=192, enc_depth=6,
                  enc_heads=6, dec_dim=128, dec_depth=2, dec_heads=4),
}[MODE]
MODEL_SIZES = {
    "baseline": None,           # 기존 smoke/pilot 크기 유지
    "stmem-small": dict(enc_dim=384, enc_depth=12, enc_heads=6,
                         dec_dim=256, dec_depth=4, dec_heads=4),
}
if MODEL_SIZE not in MODEL_SIZES:
    raise ValueError("Unknown model size")
if MODEL_SIZES[MODEL_SIZE] is not None:
    PROFILE.update(MODEL_SIZES[MODEL_SIZE])
ARCHITECTURE = {key: PROFILE[key] for key in
                ["enc_dim", "enc_depth", "enc_heads", "dec_dim", "dec_depth", "dec_heads"]}
if INITIALIZATION not in {"legacy", "independent"}:
    raise ValueError("Unknown initialization policy")
PROFILE["initialization"] = INITIALIZATION
if STEPS_OVERRIDE is not None:
    if not isinstance(STEPS_OVERRIDE, int) or STEPS_OVERRIDE < 1:
        raise ValueError("STEPS_OVERRIDE must be a positive integer.")
    PROFILE["steps"] = STEPS_OVERRIDE
GRID = [(125, 0.75)] if MODE == "smoke" else [
    (p, r) for p in PATCH_LENGTHS for r in MASK_RATIOS]
print("입력 경로:", ARTIFACT_DIR)
print("모드:", MODE, "| 모델 크기:", MODEL_SIZE, "| 학습:", RUN_TRAINING, "| 조건 수:", len(GRID))
display(pd.Series(PROFILE, name=MODE).to_frame())


## 3. 기존 산출물과 환자 분리 확인

메모리맵으로 열어 전체 약 2.6 GB를 RAM에 복사하지 않습니다.
ECG 순서/라벨/lead 순서/manifest 해시를 검증합니다.
같은 subset을 모든 조건에 재사용하며 ECG ID를 결과에 저장합니다.
Train subset은 고정 seed로 추출한 ECG 표본이며 환자·라벨 stratification 표본은 아닙니다.
Validation은 pilot에서는 공식 fold 9 전체, smoke에서는 작은 표본입니다.


In [ ]:
LEADS = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]
CLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]
manifest_path = ARTIFACT_DIR / "manifest.json"
manifest_bytes = manifest_path.read_bytes()
SOURCE_MANIFEST_SHA256 = hashlib.sha256(manifest_bytes).hexdigest()
manifest = json.loads(manifest_bytes)
FS = int(manifest["config"]["target_sampling_rate"])
assert FS == 250 and manifest["lead_order"] == LEADS
assert manifest["class_order"] == CLASSES
N_LEADS, SIGNAL_LENGTH = 12, FS * 10
arrays, frames = {}, {}
for split in ["train", "validation"]:
    x = np.load(ARTIFACT_DIR / f"signals_{split}.npy", mmap_mode="r")
    y = np.load(ARTIFACT_DIR / f"labels_{split}.npy", mmap_mode="r")
    valid = np.load(ARTIFACT_DIR / f"label_valid_{split}.npy", mmap_mode="r")
    frame = pd.read_csv(ARTIFACT_DIR / f"metadata_{split}.csv", low_memory=False)
    assert x.shape == (len(frame), N_LEADS, SIGNAL_LENGTH) and x.dtype == np.float32
    assert y.shape == (len(frame), len(CLASSES)) and np.isin(y, [0, 1]).all()
    assert valid.dtype == np.bool_ and valid.shape == (len(frame),)
    assert np.array_equal(valid, y.any(axis=1))
    assert np.array_equal(y, frame[CLASSES].to_numpy(dtype=np.uint8))
    assert np.array_equal(frame.artifact_index, np.arange(len(frame)))
    assert frame.ecg_id.is_unique and frame.split.eq(split).all()
    assert frame.strat_fold.isin(range(1, 9) if split == "train" else [9]).all()
    assert len(frame) == manifest["splits"][split]["count"]
    arrays[split] = dict(x=x, y=y, valid=valid)
    frames[split] = frame
assert not set(frames["train"].patient_id) & set(frames["validation"].patient_id)
assert not set(frames["train"].ecg_id) & set(frames["validation"].ecg_id)

def fixed_subset(n_total, limit, seed):
    if limit is None or limit >= n_total:
        return np.arange(n_total)
    return np.sort(np.random.default_rng(seed).choice(n_total, limit, replace=False))

TRAIN_IDX = fixed_subset(len(frames["train"]), PROFILE["train_n"], DATA_SEED)
VAL_IDX = fixed_subset(len(frames["validation"]), PROFILE["val_n"], DATA_SEED + 1)
PROBE_TRAIN_IDX = TRAIN_IDX[arrays["train"]["valid"][TRAIN_IDX]]
PROBE_VAL_IDX = VAL_IDX[arrays["validation"]["valid"][VAL_IDX]]
assert len(PROBE_TRAIN_IDX) and len(PROBE_VAL_IDX)
display(pd.DataFrame({
    "split": ["train", "validation"],
    "SSL_ECGs": [len(TRAIN_IDX), len(VAL_IDX)],
    "labeled_ECGs": [len(PROBE_TRAIN_IDX), len(PROBE_VAL_IDX)],
}))
display(pd.Series(manifest["config"], name="전처리 설정").to_frame())


## 4. Patchify / unpatchify와 정확한 mask 개수

한 patch는 한 lead의 겹치지 않는 P samples입니다. Flatten 순서는 lead-major:
token_id = lead_index × patches_per_lead + time_index입니다.
Tensor/array 형태는 (batch, 12×2500/P, P). 2500을 나누어떨어지게 하므로 padding은 없습니다.

Mask=True는 가린 patch입니다. 전체 격자에서 비복원 추출하므로 lead별 mask 비율을 강제로 맞추지 않습니다.
한 번에 12개 lead를 같은 시간에 가리는 temporal masking과는 다릅니다.


In [ ]:
def patchify_np(x, patch_length):
    x = np.asarray(x)
    if x.ndim != 3 or x.shape[1] != N_LEADS or x.shape[2] != SIGNAL_LENGTH:
        raise ValueError("입력은 (batch, 12, 2500)이어야 합니다.")
    if not isinstance(patch_length, (int, np.integer)) or patch_length <= 0:
        raise ValueError("patch_length는 양의 정수여야 합니다.")
    if SIGNAL_LENGTH % patch_length:
        raise ValueError("patch_length는 2500을 나누어떨어지게 해야 합니다.")
    return x.reshape(len(x), N_LEADS * (SIGNAL_LENGTH // patch_length), patch_length)

def unpatchify_np(patches, patch_length):
    patches = np.asarray(patches)
    if patches.ndim != 3 or patches.shape[1:] != (
            N_LEADS * (SIGNAL_LENGTH // patch_length), patch_length):
        raise ValueError("patch shape 불일치")
    return patches.reshape(len(patches), N_LEADS, SIGNAL_LENGTH)

def random_mask_np(batch_size, n_tokens, ratio, seed):
    if batch_size <= 0 or n_tokens < 2 or not 0 < ratio < 1:
        raise ValueError("batch>0, tokens>=2, 0<ratio<1이 필요합니다.")
    n_mask = int(round(n_tokens * ratio))
    if not 0 < n_mask < n_tokens:
        raise ValueError("visible/hidden token이 모두 있어야 합니다.")
    order = np.argsort(np.random.default_rng(seed).random((batch_size, n_tokens)), axis=1)
    mask = np.zeros((batch_size, n_tokens), dtype=bool)
    np.put_along_axis(mask, order[:, :n_mask], True, axis=1)
    return mask

def assert_patch_contract():
    sample = np.asarray(arrays["train"]["x"][TRAIN_IDX[:2]]).copy()
    assert np.isfinite(sample).all()
    for p in PATCH_LENGTHS:
        patches = patchify_np(sample, p)
        assert np.array_equal(unpatchify_np(patches, p), sample)
        # lead/time alignment: lead II의 두 번째 patch
        q = SIGNAL_LENGTH // p
        assert np.array_equal(patches[:, q + 1], sample[:, 1, p:2*p])
        for ratio in MASK_RATIOS:
            mask = random_mask_np(len(sample), patches.shape[1], ratio, SEED)
            assert (mask.sum(axis=1) == round(patches.shape[1] * ratio)).all()
            assert np.array_equal(mask, random_mask_np(len(sample), patches.shape[1], ratio, SEED))
    print("PASS: patch 원상복원 / lead-time 정렬 / 9조건 mask 개수 / seed 재현성")
assert_patch_contract()

grid_rows = []
for p in PATCH_LENGTHS:
    n = N_LEADS * SIGNAL_LENGTH // p
    for r in MASK_RATIOS:
        m = round(n * r)
        grid_rows.append(dict(patch_samples=p, patch_seconds=p/FS, mask_ratio=r,
                              total_tokens=n, masked_tokens=m, visible_tokens=n-m))
display(pd.DataFrame(grid_rows))


## 5. 9개 mask 격자와 실제 파형

검정은 가린 patch, 흰색은 보이는 patch입니다. 모든 조건에서 같은 ECG를 사용합니다.
아래 파형에서는 가린 신호를 NaN으로 표시합니다. 이는 시각화용이며 모델에는 visible token만 전달합니다.
가린 값을 0으로 덮고 일반 convolution을 통과시키는 구현이 아닙니다.


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(13, 8), sharey=True)
for i, p in enumerate(PATCH_LENGTHS):
    for j, r in enumerate(MASK_RATIOS):
        n_time = SIGNAL_LENGTH // p
        mask = random_mask_np(1, N_LEADS*n_time, r, SEED)[0].reshape(N_LEADS, n_time)
        ax = axes[i, j]
        ax.imshow(mask, aspect="auto", cmap="Greys", vmin=0, vmax=1,
                  extent=[0, 10, 11.5, -0.5], interpolation="nearest")
        ax.set_title(f"P={p} ({p/FS:.1f}s), mask={r:.0%}")
        ax.set_yticks(range(N_LEADS), LEADS)
        if i == 2: ax.set_xlabel("Time (s)")
fig.suptitle("Black = masked; white = visible")
plt.tight_layout()
plt.show()
plt.close(fig)

p, r = 125, 0.75
example = np.asarray(arrays["train"]["x"][TRAIN_IDX[0]]).copy()
mask = random_mask_np(1, N_LEADS*SIGNAL_LENGTH//p, r, SEED)[0].reshape(N_LEADS, -1)
sample_mask = np.repeat(mask, p, axis=1)
lead = LEADS.index("II")
visible = example[lead].copy()
visible[sample_mask[lead]] = np.nan
fig, ax = plt.subplots(figsize=(13, 3))
ax.plot(np.arange(SIGNAL_LENGTH)/FS, example[lead], color="lightgray", label="Target")
ax.plot(np.arange(SIGNAL_LENGTH)/FS, visible, label="Visible")
ax.set(title=f"ECG {int(frames['train'].iloc[TRAIN_IDX[0]].ecg_id)}, lead II",
       xlabel="Time (s)", ylabel="Normalized amplitude")
ax.legend()
plt.tight_layout()
plt.show()
plt.close(fig)


## 6. Plain ECG-MAE: visible-only encoder + 작은 decoder

Encoder는 보이는 patch만 받습니다. Decoder에서 원래 lead/time 위치에 mask token을 추가합니다.
Lead와 time embedding을 분리하고, decoder는 전체 lead/time token에 attention합니다.
ST-MEM의 lead-wise decoder를 복제한 모델은 아닙니다.

Loss는 masked sample의 평균 MSE입니다. 현재 target은 ECG별·lead별 정규화된 파형입니다.
추가 patch별 정규화는 하지 않습니다. constant/near-flat lead는 전처리 정책대로 유지하므로
난이도 estimator를 구현할 때 이들의 처리 정책을 따로 정해야 합니다.


In [ ]:
if TRAINING_AVAILABLE:
    def seed_everything(seed):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False

    def patchify_torch(x, p):
        assert x.ndim == 3 and x.shape[1:] == (N_LEADS, SIGNAL_LENGTH)
        assert SIGNAL_LENGTH % p == 0
        return x.reshape(x.shape[0], N_LEADS * SIGNAL_LENGTH // p, p)

    def masked_mse(pred, target, mask):
        if pred.shape != target.shape or mask.shape != pred.shape[:2]:
            raise ValueError("prediction/target/mask shape 불일치")
        if mask.dtype != torch.bool or not mask.any():
            raise ValueError("mask는 가린 위치가 있는 bool tensor여야 합니다.")
        return ((pred - target).square().mean(dim=-1) * mask).sum() / mask.sum()

    class PlainECGMAE(nn.Module):
        def __init__(self, patch_length, cfg):
            super().__init__()
            # Old checkpoint profiles have no policy key and retain legacy initialization.
            self.initialization = cfg.get("initialization", "legacy")
            if self.initialization not in {"legacy", "independent"}:
                raise ValueError("Unknown initialization policy")
            self.p = patch_length
            self.q = SIGNAL_LENGTH // patch_length
            self.n = N_LEADS * self.q
            e, d = cfg["enc_dim"], cfg["dec_dim"]
            self.patch_embed = nn.Linear(patch_length, e)
            self.enc_lead = nn.Parameter(torch.randn(1, N_LEADS, 1, e) * 0.02)
            self.enc_time = nn.Parameter(torch.randn(1, 1, self.q, e) * 0.02)
            layer = nn.TransformerEncoderLayer(e, cfg["enc_heads"], e*4,
                        dropout=0.0, activation="gelu", batch_first=True, norm_first=True)
            self.encoder = nn.TransformerEncoder(layer, cfg["enc_depth"],
                                                  norm=nn.LayerNorm(e), enable_nested_tensor=False)
            self.to_decoder = nn.Linear(e, d)
            self.mask_token = nn.Parameter(torch.randn(1, 1, d) * 0.02)
            self.dec_lead = nn.Parameter(torch.randn(1, N_LEADS, 1, d) * 0.02)
            self.dec_time = nn.Parameter(torch.randn(1, 1, self.q, d) * 0.02)
            dec = nn.TransformerEncoderLayer(d, cfg["dec_heads"], d*4,
                        dropout=0.0, activation="gelu", batch_first=True, norm_first=True)
            self.decoder = nn.TransformerEncoder(dec, cfg["dec_depth"],
                                                  norm=nn.LayerNorm(d), enable_nested_tensor=False)
            self.reconstruction = nn.Linear(d, patch_length)
            if self.initialization == "independent":
                # Reinitialize only Transformer blocks; preserve embeddings,
                # reconstruction head, architecture, mask policy and optimizer.
                for block in [*self.encoder.layers, *self.decoder.layers]:
                    block.apply(self._initialize_transformer_module)

        @staticmethod
        def _initialize_transformer_module(module):
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.MultiheadAttention):
                nn.init.xavier_uniform_(module.in_proj_weight)
                if module.in_proj_bias is not None:
                    nn.init.zeros_(module.in_proj_bias)
            elif isinstance(module, nn.LayerNorm):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)

        def embedded_patches(self, x):
            patches = patchify_torch(x, self.p)
            position = (self.enc_lead + self.enc_time).reshape(1, self.n, -1)
            return patches, self.patch_embed(patches) + position

        def forward(self, x, mask):
            patches, tokens = self.embedded_patches(x)
            counts = (~mask).sum(dim=1)
            if mask.dtype != torch.bool or mask.shape != tokens.shape[:2]:
                raise ValueError("mask shape/dtype 불일치")
            if not (counts == counts[0]).all() or not 0 < counts[0] < self.n:
                raise ValueError("batch의 visible 개수는 같고 0<Nvisible<N이어야 합니다.")
            # Visible 위치를 시간/lead 순서대로 gather합니다.
            ids = torch.arange(self.n, device=x.device).expand(len(x), -1)
            visible_ids = ids[~mask].reshape(len(x), int(counts[0]))
            visible = tokens.gather(1, visible_ids[..., None].expand(-1, -1, tokens.shape[-1]))
            encoded = self.encoder(visible)
            decoded_visible = self.to_decoder(encoded)
            full = self.mask_token.expand(len(x), self.n, -1).clone()
            full.scatter_(1, visible_ids[..., None].expand(-1, -1, full.shape[-1]), decoded_visible)
            full = full + (self.dec_lead + self.dec_time).reshape(1, self.n, -1)
            return self.reconstruction(self.decoder(full)), patches

        def features(self, x):
            _, tokens = self.embedded_patches(x)
            return self.encoder(tokens).mean(dim=1)


## 7. CPU/GPU 안전장치와 모델 단위 검사

CPU에서도 학습 가능하지만 token 600개의 full attention decoder와 반복 실험은 비쌉니다.
GPU pilot은 별도 모델이 아니라 모든 9조건에 같은 encoder/decoder 크기를 사용합니다.
Patch embedding과 출력 층의 크기, position embedding 수만 patch length에 따라 달라집니다.
동일 update/ECG 노출 수는 동일 FLOPs가 아닙니다. 시간·메모리를 별도 보고하세요.
GPU/CPU 및 PyTorch 버전 간 완전 동일한 수치 재현성을 보장하지 않습니다.


In [ ]:
DEVICE = None
if TRAINING_AVAILABLE:
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if DEVICE.type == "cpu":
        torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))
    seed_everything(SEED)
    model = PlainECGMAE(125, PROFILE).to(DEVICE)
    x = torch.from_numpy(np.asarray(arrays["train"]["x"][TRAIN_IDX[:2]]).copy()).to(DEVICE)
    mask = torch.from_numpy(random_mask_np(len(x), model.n, 0.75, SEED)).to(DEVICE)
    pred, target = model(x, mask)
    loss = masked_mse(pred, target, mask)
    assert pred.shape == (len(x), 240, 125) and torch.isfinite(loss)
    loss.backward()
    assert all(p.grad is None or torch.isfinite(p.grad).all() for p in model.parameters())
    # Visible-only loss 제외 및 가린 target 값이 encoder 입력에 유입되지 않는지 확인합니다.
    model.eval()
    with torch.no_grad():
        # train/grad 경로와 eval/no-grad fast path의 수치 차이를 비교하지 않습니다.
        pred_reference, target_reference = model(x, mask)
        reference_loss = masked_mse(pred_reference, target_reference, mask)
        changed = x.clone()
        changed.reshape(len(x), model.n, model.p)[mask] += 100
        pred_changed, _ = model(changed, mask)
        assert torch.allclose(pred_reference, pred_changed, atol=1e-5, rtol=1e-5), (
            f"Hidden-input isolation failed: max_abs_diff={(pred_reference-pred_changed).abs().max().item():.3e}")
        altered = pred_reference.clone()
        altered[~mask] += 100
        assert torch.allclose(masked_mse(altered, target_reference, mask), reference_loss), (
            "Visible predictions must not affect masked-only loss.")
    print("PASS: forward/backward finite / hidden-input isolation / masked-only loss")
    print("device:", DEVICE, "| parameters:", sum(p.numel() for p in model.parameters()))
    del model, x, mask, pred, target, loss, changed, pred_changed, altered
    del pred_reference, target_reference, reference_loss
else:
    print("모델 검사는 생략했습니다. 학습 라이브러리를 설치하면 실행됩니다.")

if RUN_TRAINING:
    if not TRAINING_AVAILABLE:
        raise RuntimeError("학습을 위해 torch와 scikit-learn을 설치하세요.")
    if MODE == "pilot" and DEVICE.type != "cuda" and not ALLOW_CPU_PILOT:
        raise RuntimeError("GPU pilot은 CUDA GPU에서 실행하세요. CPU에서는 MODE='smoke'를 사용하세요.")


## 8. 고정 update 학습 / 고정 validation masks / frozen linear probe

모든 조건에서 같은 ECG subset, seed, batch 크기, update 수를 사용합니다.
학습 마스크는 매 update 새로 샘플링합니다. 검증 마스크는 ECG ID로 seed를 만들고 3회 평균합니다.
각 조건의 최종 checkpoint를 평가하며 validation으로 encoder checkpoint를 선택하지 않습니다.
진단은 기본 활성화입니다. 모델/optimizer/mask 정책은 변경하지 않고 0·200·400...·최종 update에서
고정 train/validation 32 ECG와 고정 mask 1회를 평가합니다(smoke도 0·최종 포함).
주 결과의 validation MSE는 기존처럼 최대 128 ECG·고정 mask 3회로 계산합니다.
진단 곡선은 작은 subset 관찰용이며, 최종 평가나 best checkpoint 선택을 대체하지 않습니다.
계산/그림 함수는 같은 폴더의 masking_diagnostics.py에 분리했습니다. 서버에도 함께 복사하세요.

Linear probe는 encoder를 eval/freeze하고 모든 patch 표현을 평균 pooling한 뒤 선형층만 학습합니다.
특징 표준화 통계는 probe train에서만 계산합니다. 라벨 없는 ECG는 SSL에는 포함, probe에서는 제외합니다.
Macro AUROC/AUPRC와 클래스별 지표를 기록합니다. F1 threshold는 이번 pilot에서 고정 0.5입니다.
Validation 단일 라벨만 존재하는 클래스는 AUROC/AUPRC에서 제외하고 개수를 명시합니다.
모든 클래스가 유효하지 않은 smoke 결과는 순위 결정에 쓰지 않습니다.


In [ ]:
def save_learning_curves(history, condition_dir, patch_length, ratio, window=50):
    # Visualization only: do not change history, model state, or sampling RNGs.
    if not isinstance(window, (int, np.integer)) or window < 1:
        raise ValueError("Moving-average window must be a positive integer.")
    columns = ["step", "masked_mse", "grad_norm", "lr"]
    logged = history.loc[:, columns].sort_values("step")
    if logged.empty or not np.isfinite(logged.to_numpy()).all():
        raise ValueError("Training history must be non-empty and finite.")
    width = min(window, len(logged))
    smoothed = logged.masked_mse.rolling(width, min_periods=1).mean()
    destination = Path(condition_dir) / "learning_curves.png"
    destination.parent.mkdir(parents=True, exist_ok=True)
    fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True,
                             constrained_layout=True)
    try:
        axes[0].plot(logged.step, logged.masked_mse, color="C0", alpha=0.25,
                     linewidth=0.8, label="Raw batch loss")
        axes[0].plot(logged.step, smoothed, color="C0", linewidth=1.8,
                     label=f"Trailing mean ({width} updates, partial window at start)")
        axes[0].set_ylabel("Training masked MSE")
        axes[0].set_title("Raw training batches (not fixed-mask train/validation evaluation)",
                          fontsize=10)
        axes[0].legend(loc="best", fontsize=9)
        axes[1].plot(logged.step, logged.grad_norm, color="C1",
                     label="Total gradient norm before clipping")
        axes[1].axhline(1.0, color="0.4", linestyle="--", linewidth=1,
                       label="Clipping threshold (1.0)")
        axes[1].set_ylabel("Gradient L2 norm")
        axes[1].legend(loc="best", fontsize=9)
        axes[2].plot(logged.step, logged.lr, color="C2")
        axes[2].set_ylabel("Logged learning rate")
        axes[2].set_title("LR logged after scheduler.step(): rate for the next update",
                          fontsize=10)
        axes[2].ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
        axes[2].set_xlabel("Training update")
        for ax in axes:
            ax.grid(alpha=0.2)
        fig.suptitle(f"MAE pretraining | patch {patch_length} samples | mask {ratio:.0%}")
        fig.savefig(destination, dpi=160)
    finally:
        plt.close(fig)
    return destination


if TRAINING_AVAILABLE:
    diagnostic_path = next((p / "masking_diagnostics.py" for p in
                            [cwd, cwd / "experiments", REPO_DIR / "experiments"]
                            if (p / "masking_diagnostics.py").is_file()), None)
    if diagnostic_path is None:
        raise FileNotFoundError("Copy masking_diagnostics.py alongside Masking_pilot.ipynb.")
    diagnostic_spec = importlib.util.spec_from_file_location("masking_diagnostics", diagnostic_path)
    diagnostics = importlib.util.module_from_spec(diagnostic_spec)
    diagnostic_spec.loader.exec_module(diagnostics)
    TRAIN_DIAGNOSTIC_IDX = TRAIN_IDX[fixed_subset(len(TRAIN_IDX), DIAGNOSTICS["eval_n"], DATA_SEED + 2)]
    VAL_DIAGNOSTIC_IDX = VAL_IDX[:min(DIAGNOSTICS["eval_n"], PROFILE["eval_n"], len(VAL_IDX))]
    LINEAR_FIT_IDX = TRAIN_IDX[fixed_subset(len(TRAIN_IDX), DIAGNOSTICS["cross_lead_fit_n"], DATA_SEED + 4)]
    LINEAR_BASELINE = None

    def read_batch(split, indices, device):
        x = np.asarray(arrays[split]["x"][indices], dtype=np.float32).copy()
        if not np.isfinite(x).all():
            raise ValueError("신호에 NaN/Inf가 있습니다.")
        return torch.from_numpy(x).to(device)

    @torch.no_grad()
    def reconstruction_eval(model, indices, batch_size):
        model.eval()
        total, count = 0.0, 0
        for repeat in range(3):
            for start in range(0, len(indices), batch_size):
                selected = indices[start:start + batch_size]
                masks = [random_mask_np(1, model.n, model.ratio,
                           EVAL_SEED + 100000 + repeat*1000000 +
                           int(frames["validation"].iloc[idx].ecg_id))[0] for idx in selected]
                mask = torch.from_numpy(np.stack(masks)).to(DEVICE)
                x = read_batch("validation", selected, DEVICE)
                pred, target = model(x, mask)
                error = (pred - target).square().mean(dim=-1)
                total += float((error * mask).sum())
                count += int(mask.sum())
        return total / count

    @torch.no_grad()
    def extract_features(model, split, indices):
        model.eval()
        features = []
        for start in range(0, len(indices), PROFILE["batch"]):
            x = read_batch(split, indices[start:start+PROFILE["batch"]], DEVICE)
            features.append(model.features(x).cpu())
        return torch.cat(features)

    def classification_metrics(labels, probabilities):
        rows = []
        for j, name in enumerate(CLASSES):
            y, score = labels[:, j], probabilities[:, j]
            both = len(np.unique(y)) == 2
            rows.append(dict(
                class_name=name, positives=int(y.sum()), negatives=int(len(y)-y.sum()),
                auroc=float(roc_auc_score(y, score)) if both else np.nan,
                auprc=float(average_precision_score(y, score)) if both else np.nan,
                f1=float(f1_score(y, score >= 0.5, zero_division=0)),
            ))
        detail = pd.DataFrame(rows)
        metrics = dict(macro_auroc=float(detail.auroc.mean()),
                       macro_auprc=float(detail.auprc.mean()),
                       macro_f1=float(detail.f1.mean()),
                       valid_metric_classes=int(detail.auroc.notna().sum()))
        return metrics, detail

    def fit_linear_probe(model):
        for parameter in model.parameters():
            parameter.requires_grad_(False)
        train_features = extract_features(model, "train", PROBE_TRAIN_IDX)
        val_features = extract_features(model, "validation", PROBE_VAL_IDX)
        feature_diagnostics = (diagnostics.feature_statistics(train_features)
                               if DIAGNOSTICS["enabled"] else None)
        mean = train_features.mean(dim=0, keepdim=True)
        std = train_features.std(dim=0, unbiased=False, keepdim=True).clamp_min(1e-6)
        train_features, val_features = (train_features-mean)/std, (val_features-mean)/std
        labels = torch.from_numpy(np.asarray(arrays["train"]["y"][PROBE_TRAIN_IDX],
                                             dtype=np.float32).copy())
        seed_everything(SEED + 100)
        head = nn.Linear(train_features.shape[1], len(CLASSES)).to(DEVICE)
        optimizer = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=1e-4)
        criterion = nn.BCEWithLogitsLoss()
        rng = np.random.default_rng(SEED + 100)
        head.train()
        for _ in range(PROFILE["probe_steps"]):
            chosen = rng.choice(len(labels), min(PROFILE["probe_batch"], len(labels)), replace=False)
            logits = head(train_features[chosen].to(DEVICE))
            loss = criterion(logits, labels[chosen].to(DEVICE))
            if not torch.isfinite(loss):
                raise RuntimeError("probe loss NaN/Inf")
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
        head.eval()
        with torch.no_grad():
            probabilities = torch.cat([
                head(val_features[start:start+PROFILE["probe_batch"]].to(DEVICE)).sigmoid().cpu()
                for start in range(0, len(val_features), PROFILE["probe_batch"])
            ]).numpy()
        y = np.asarray(arrays["validation"]["y"][PROBE_VAL_IDX])
        metrics, detail = classification_metrics(y, probabilities)
        state = dict(head={k: v.detach().cpu() for k, v in head.state_dict().items()},
                     feature_mean=mean, feature_std=std, threshold=0.5,
                     feature_diagnostics=feature_diagnostics)
        return metrics, detail, probabilities, state

    def fixed_diagnostic_evaluation(model, step):
        rows, bundles = [], {}
        for split, indices in [("train", TRAIN_DIAGNOSTIC_IDX), ("validation", VAL_DIAGNOSTIC_IDX)]:
            bundle = diagnostics.evaluate_reconstruction(
                model, arrays[split]["x"], frames[split].ecg_id.to_numpy(),
                indices, PROFILE["batch"], EVAL_SEED, repeats=DIAGNOSTICS["repeats"],
                examples=DIAGNOSTICS["examples"] if split == "validation" and step == 0 else 0,
                lead_order=LEADS, fs=FS)
            rows.append(dict(step=step, split=split, ecgs=len(indices),
                             repeats=DIAGNOSTICS["repeats"], **bundle["metrics"]))
            bundles[split] = bundle
        return rows, bundles

    def train_condition(patch_length, ratio, condition_dir=None):
        global LINEAR_BASELINE
        condition_started = time.perf_counter()
        seed_everything(SEED)
        model = PlainECGMAE(patch_length, PROFILE).to(DEVICE)
        model.ratio = ratio
        diagnostic_enabled = DIAGNOSTICS["enabled"]
        if diagnostic_enabled and any(DIAGNOSTICS[k] < 1 for k in
                                     ["eval_every", "eval_n", "repeats", "grad_every", "cross_lead_fit_n"]):
            raise ValueError("Diagnostic intervals/sizes must be positive.")
        diagnostic_dir = Path(condition_dir) if condition_dir is not None else None
        if diagnostic_dir is not None:
            diagnostic_dir.mkdir(parents=True, exist_ok=True)
        initial_state = ({k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                         if diagnostic_enabled and DIAGNOSTICS["random_encoder_probe"] else None)
        evaluations, gradient_rows = [], []
        initial_diagnostic_seconds = 0.0
        if diagnostic_enabled:
            diagnostic_started = time.perf_counter()
            if LINEAR_BASELINE is None:
                LINEAR_BASELINE = diagnostics.fit_cross_lead_baseline(
                    arrays["train"]["x"], LINEAR_FIT_IDX, ridge=DIAGNOSTICS["cross_lead_ridge"])
            rows, initial_bundles = fixed_diagnostic_evaluation(model, 0)
            evaluations.extend(rows)
            if diagnostic_dir is not None:
                diagnostics.write_json(diagnostic_dir / "initialization.json", diagnostics.initialization_report(model))
                np.savez_compressed(diagnostic_dir / "cross_lead_baseline.npz",
                                    **LINEAR_BASELINE, lead_order=np.asarray(LEADS))
                if initial_state is not None:
                    torch.save(dict(model=initial_state, patch_length=patch_length, profile=PROFILE, model_size=MODEL_SIZE,
                                    mask_ratio=ratio, seed=SEED, source_manifest_sha256=SOURCE_MANIFEST_SHA256),
                               diagnostic_dir / "initial_checkpoint.pt")
                diagnostics.save_reconstruction(initial_bundles["validation"], diagnostic_dir, prefix="initial")
                diagnostics.save_diagnostic_progress(evaluations, gradient_rows, diagnostic_dir, plot=False)
            del initial_bundles
            initial_diagnostic_seconds = time.perf_counter() - diagnostic_started
        optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4,
                                       betas=(0.9, 0.95), weight_decay=0.05)
        warmup = min(100, max(1, PROFILE["steps"] // 10))
        def schedule(step):
            if step < warmup:
                return (step+1)/warmup
            fraction = (step-warmup)/max(1, PROFILE["steps"]-warmup)
            return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi*fraction))
        scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, schedule)
        rng = np.random.default_rng(SEED)
        history = []
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.synchronize()
        started = time.perf_counter()
        periodic_diagnostic_seconds = 0.0
        model.train()
        for step in range(PROFILE["steps"]):
            indices = rng.choice(TRAIN_IDX, min(PROFILE["batch"], len(TRAIN_IDX)), replace=False)
            x = read_batch("train", indices, DEVICE)
            mask = torch.from_numpy(random_mask_np(len(x), model.n, ratio,
                                                   SEED + 1000 + step)).to(DEVICE)
            pred, target = model(x, mask)
            loss = masked_mse(pred, target, mask)
            if not torch.isfinite(loss):
                raise RuntimeError("MAE loss NaN/Inf")
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            sample_gradients = diagnostic_enabled and (step == 0 or
                (step+1) % DIAGNOSTICS["grad_every"] == 0 or step+1 == PROFILE["steps"])
            if sample_gradients:
                gradient_snapshot, weights_before = diagnostics.gradient_snapshot(model, step+1)
            norm = nn.utils.clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=True)
            lr_used = optimizer.param_groups[0]["lr"]
            optimizer.step()
            if sample_gradients:
                gradient_rows.extend(diagnostics.complete_update_snapshot(model, gradient_snapshot, weights_before))
                del weights_before, gradient_snapshot
            scheduler.step()
            history.append(dict(step=step+1, masked_mse=float(loss.detach()),
                                grad_norm=float(norm), lr=optimizer.param_groups[0]["lr"]))
            if diagnostic_enabled:
                with torch.no_grad():
                    zero_mse = float((target.square().mean(dim=-1)*mask).sum()/mask.sum())
                    pred_rms = float(((pred.detach().square().mean(dim=-1)*mask).sum()/mask.sum()).sqrt())
                history[-1].update(zero_masked_mse=zero_mse,
                                   skill_vs_zero=1-history[-1]["masked_mse"]/zero_mse if zero_mse > 0 else np.nan,
                                   pred_rms_masked=pred_rms, target_rms_masked=math.sqrt(zero_mse),
                                   lr_used=lr_used, clipped=float(norm) > 1.0)
                if (step+1) % DIAGNOSTICS["eval_every"] == 0 or step+1 == PROFILE["steps"]:
                    diagnostic_started = time.perf_counter()
                    rows, _ = fixed_diagnostic_evaluation(model, step+1)
                    evaluations.extend(rows)
                    if diagnostic_dir is not None:
                        pd.DataFrame(history).to_csv(diagnostic_dir / "pretrain_history.csv", index=False)
                        diagnostics.save_diagnostic_progress(evaluations, gradient_rows, diagnostic_dir, plot=False)
                    print(f"Diagnostic update {step+1}: train={rows[0]['masked_mse']:.5f}, "
                          f"val={rows[1]['masked_mse']:.5f}, val skill={rows[1]['skill_vs_zero']:.2%}", flush=True)
                    periodic_diagnostic_seconds += time.perf_counter() - diagnostic_started
            if (step+1) % max(1, PROFILE["steps"]//5) == 0:
                print(f"P={patch_length}, r={ratio}: {step+1}/{PROFILE['steps']}, MSE={history[-1]['masked_mse']:.5f}",
                      flush=True)
        if DEVICE.type == "cuda": torch.cuda.synchronize()
        pretrain_wall_seconds = time.perf_counter() - started
        train_seconds = pretrain_wall_seconds - periodic_diagnostic_seconds
        eval_indices = VAL_IDX[:min(PROFILE["eval_n"], len(VAL_IDX))]
        final_diagnostic_seconds, random_probe_seconds = 0.0, 0.0
        random_metrics, random_state = None, None
        if diagnostic_enabled:
            diagnostic_started = time.perf_counter()
            final_bundle = diagnostics.evaluate_reconstruction(
                model, arrays["validation"]["x"], frames["validation"].ecg_id.to_numpy(),
                eval_indices, PROFILE["batch"], EVAL_SEED, repeats=3, examples=DIAGNOSTICS["examples"],
                lead_order=LEADS, fs=FS, linear_baseline=LINEAR_BASELINE)
            mse = final_bundle["metrics"]["masked_mse"]
            if diagnostic_dir is not None:
                diagnostics.save_reconstruction(final_bundle, diagnostic_dir, prefix="final")
                diagnostics.save_diagnostic_progress(evaluations, gradient_rows, diagnostic_dir)
            final_diagnostic_seconds = time.perf_counter() - diagnostic_started
            if initial_state is not None:
                random_started = time.perf_counter()
                with diagnostics.preserve_rng(DEVICE):
                    random_model = PlainECGMAE(patch_length, PROFILE).to(DEVICE)
                    random_model.load_state_dict(initial_state)
                    random_metrics, random_detail, random_probabilities, random_state = fit_linear_probe(random_model)
                    if diagnostic_dir is not None:
                        diagnostics.write_json(diagnostic_dir / "random_encoder_probe_metrics.json", random_metrics)
                        random_detail.to_csv(diagnostic_dir / "random_encoder_class_metrics.csv", index=False)
                        np.save(diagnostic_dir / "random_encoder_probabilities.npy", random_probabilities)
                        torch.save(random_state, diagnostic_dir / "random_encoder_probe.pt")
                    del random_model, random_detail, random_probabilities
                random_probe_seconds = time.perf_counter() - random_started
                del initial_state
        else:
            mse = reconstruction_eval(model, eval_indices, PROFILE["batch"])
        probe_started = time.perf_counter()
        metrics, detail, probabilities, probe_state = fit_linear_probe(model)
        if DEVICE.type == "cuda": torch.cuda.synchronize()
        probe_seconds = time.perf_counter()-probe_started
        if diagnostic_enabled and diagnostic_dir is not None:
            diagnostics.write_json(diagnostic_dir / "feature_statistics.json",
                dict(trained=probe_state["feature_diagnostics"],
                     random=None if random_state is None else random_state["feature_diagnostics"]))
            diagnostics.write_json(diagnostic_dir / "diagnostic_summary.json",
                diagnostics.diagnostic_summary(final_bundle["metrics"], pd.DataFrame(history),
                                               random_metrics, metrics, evaluations))
        result = dict(mode=MODE, model_size=MODEL_SIZE, seed=SEED, patch_samples=patch_length,
                      patch_seconds=patch_length/FS, mask_ratio=ratio, total_tokens=model.n,
                      visible_tokens=model.n-round(model.n*ratio),
                      parameters=sum(v.numel() for v in model.parameters()),
                      steps=PROFILE["steps"], train_ecgs=len(TRAIN_IDX),
                      labeled_train_ecgs=len(PROBE_TRAIN_IDX), validation_ecgs=len(PROBE_VAL_IDX),
                      validation_masked_mse=mse, train_seconds=train_seconds,
                      probe_seconds=probe_seconds, pretrain_wall_seconds=pretrain_wall_seconds,
                      periodic_diagnostic_seconds=periodic_diagnostic_seconds,
                      diagnostic_seconds=initial_diagnostic_seconds + periodic_diagnostic_seconds + final_diagnostic_seconds,
                      random_probe_seconds=random_probe_seconds,
                      gpu_peak_allocated_mb=(torch.cuda.max_memory_allocated()/2**20
                                             if DEVICE.type == "cuda" else None),
                      **metrics)
        if diagnostic_enabled:
            for key in ["zero_masked_mse", "skill_vs_zero", "rms_ratio", "masked_correlation", "linear_masked_mse"]:
                result[f"validation_{key}"] = final_bundle["metrics"][key]
            result["clipping_fraction"] = float(pd.DataFrame(history).clipped.mean())
            if random_metrics is not None:
                result["random_encoder_macro_auroc"] = random_metrics["macro_auroc"]
                result["auroc_gain_vs_random"] = metrics["macro_auroc"]-random_metrics["macro_auroc"]
                result["random_encoder_macro_auprc"] = random_metrics["macro_auprc"]
                result["auprc_gain_vs_random"] = metrics["macro_auprc"]-random_metrics["macro_auprc"]
                result["random_encoder_macro_f1"] = random_metrics["macro_f1"]
                result["f1_gain_vs_random"] = metrics["macro_f1"]-random_metrics["macro_f1"]
        result["condition_seconds"] = time.perf_counter() - condition_started
        return model, result, pd.DataFrame(history), detail, probabilities, probe_state


## 9. 명시적으로 켠 경우에만 실행하고 조건별 결과 저장

매 실행마다 새로운 run 폴더를 만들어 기존 실험을 덮어쓰지 않습니다.
Metadata에는 subset ECG ID, 모델/학습 설정, 전처리 manifest와 SHA256, 라이브러리 버전을 기록합니다.
각 조건의 학습·평가가 끝나면 pretrain_history.csv와 learning_curves.png를 자동 저장합니다.
PNG는 batch별 masked MSE 원본·50-update 이동평균, clipping 전 gradient norm, 학습률을 보여줍니다.
짧은 smoke는 이동평균 창을 기록 길이로 줄이고, 시작 구간은 확보된 update만 평균합니다.
CSV의 lr은 scheduler.step() 이후 값(다음 update의 학습률)입니다.
진단 활성화 시 evaluation_history.csv / diagnostic_curves.png에 고정 train/validation 곡선을 별도 저장합니다.
initial/final_reconstruction_samples.npz는 정답·모델 예측·patch/sample mask·visible-copy 합성 파형과 ECG ID를 담습니다.
비교 PNG는 가린 구간의 예측만 표시하며 진폭은 mV가 아닌 전처리 z-score입니다.
최종 validation에 같은 마스크의 zero 및 train-only cross-lead 선형 baseline을 비교합니다.
gradient_updates.csv/png는 clipping 전 module/layer gradient와 optimizer 이후 실제 변화량을 저장합니다.
같은 초기 encoder·같은 subset·probe 설정으로 random encoder 대조군도 평가합니다.
diagnostic_summary.json은 약한 복원/진폭 수축/작은 SSL 이득을 표시하되 원인을 확정하지 않습니다.
진단 평가/probe 비용 때문에 기존 실행보다 느립니다. 별도 진단 시간과 전체 condition 시간을 기록합니다.
CSV는 보기 위한 산출물이고 best 설정 자동 확정은 하지 않습니다.
중단되면 완료된 조건은 남습니다. 자동 resume은 지원하지 않습니다.


In [ ]:
RESULTS = pd.DataFrame()
RUN_DIR = None
if RUN_TRAINING:
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    RUN_DIR = OUTPUT_ROOT / f"{stamp}_{MODE}_seed{SEED}_{uuid.uuid4().hex[:8]}"
    RUN_DIR.mkdir()
    versions = {name: importlib.metadata.version(name)
                for name in ["numpy", "pandas", "matplotlib", "torch", "scikit-learn"]}
    run_config = dict(
        mode=MODE, model_size=MODEL_SIZE, architecture=ARCHITECTURE, seed=SEED, subset_seed=DATA_SEED, evaluation_mask_seed=EVAL_SEED, device=str(DEVICE), versions=versions, profile=PROFILE,
        grid=GRID, class_order=CLASSES, lead_order=LEADS, source_manifest_sha256=SOURCE_MANIFEST_SHA256,
        source_preprocessing=manifest["config"], source_manifest=manifest,
        train_ecg_ids=frames["train"].iloc[TRAIN_IDX].ecg_id.astype(int).tolist(),
        validation_ecg_ids=frames["validation"].iloc[VAL_IDX].ecg_id.astype(int).tolist(),
        reconstruction_validation_ecg_ids=frames["validation"].iloc[
            VAL_IDX[:min(PROFILE["eval_n"], len(VAL_IDX))]].ecg_id.astype(int).tolist(),
        pretrain_optimizer=dict(name="AdamW", lr=3e-4, betas=[0.9, 0.95], weight_decay=0.05,
                                warmup_steps=min(100, max(1, PROFILE["steps"]//10)),
                                schedule="warmup + cosine to 0.1*base_lr", grad_clip=1.0),
        probe_optimizer=dict(name="AdamW", lr=1e-3, weight_decay=1e-4, loss="BCEWithLogitsLoss"),
        history_logging=dict(interval_updates=1, moving_average_window=50,
                             grad_norm="before clipping", lr="after scheduler step (next update)",
                             validation="periodic fixed diagnostic subset + final 3-repeat metric"
                             if DIAGNOSTICS["enabled"] else "final checkpoint only"),
        diagnostics_config=DIAGNOSTICS,
        diagnostic_train_ecg_ids=frames["train"].iloc[TRAIN_DIAGNOSTIC_IDX].ecg_id.astype(int).tolist(),
        diagnostic_validation_ecg_ids=frames["validation"].iloc[VAL_DIAGNOSTIC_IDX].ecg_id.astype(int).tolist(),
        linear_baseline_train_ecg_ids=frames["train"].iloc[LINEAR_FIT_IDX].ecg_id.astype(int).tolist(),
        mask_seed_formula="evaluation_mask_seed + 100000 + repeat*1000000 + ecg_id; argsort RNG random values",
        diagnostic_source_sha256=hashlib.sha256(diagnostic_path.read_bytes()).hexdigest(),
        policy="global-uniform-random-exact-count", target="preprocessed waveform",
        pooling="mean-all-lead-time-tokens", threshold=0.5,
        note="smoke is not scientific evidence; pilot requires multi-seed longer confirmation",
    )
    (RUN_DIR / "run_config.json").write_text(
        json.dumps(run_config, ensure_ascii=False, indent=2), encoding="utf-8")
    records = []
    print("저장 경로:", RUN_DIR, flush=True)
    for p, ratio in GRID:
        condition_dir = RUN_DIR / f"p{p}_r{int(round(ratio*100)):02d}"
        condition_dir.mkdir()
        model, result, history, detail, probabilities, probe_state = train_condition(p, ratio, condition_dir)
        history.to_csv(condition_dir / "pretrain_history.csv", index=False)
        detail.to_csv(condition_dir / "class_metrics.csv", index=False)
        np.save(condition_dir / "validation_probabilities.npy", probabilities)
        np.save(condition_dir / "validation_ecg_ids.npy",
                frames["validation"].iloc[PROBE_VAL_IDX].ecg_id.to_numpy(dtype=np.int64))
        torch.save(dict(model={k: v.detach().cpu() for k, v in model.state_dict().items()},
                        patch_length=p, mask_ratio=ratio, seed=SEED, profile=PROFILE, model_size=MODEL_SIZE, probe=probe_state,
                        source_manifest_sha256=SOURCE_MANIFEST_SHA256),
                   condition_dir / "checkpoint.pt")
        records.append(result)
        RESULTS = pd.DataFrame(records)
        RESULTS.to_csv(RUN_DIR / "results.csv", index=False)
        curve_path = save_learning_curves(history, condition_dir, p, ratio)
        print("학습 곡선 저장:", curve_path, flush=True)
        display(RESULTS.tail(1))
        del model
        if DEVICE.type == "cuda": torch.cuda.empty_cache()
else:
    print("학습하지 않았습니다. RUN_TRAINING=True로 바꾸면 MODE에 맞는 실험이 실행됩니다.")


## 10. 결과 해석과 다음 결정

1. Smoke 결과에서는 좋은 설정을 고르지 않습니다. 오류 검증만 합니다.
2. Pilot에서는 validation macro AUROC를 우선으로 보고 AUPRC/클래스별 결과도 확인합니다.
3. 서로 다른 mask 비율·patch 길이의 reconstruction MSE는 과제 난이도가 달라 직접 우열 지표가 아닙니다.
4. 상위 두 조건을 공통 모델로 더 긴 학습과 seed 3개에서 비교합니다. 작은 차이를 단일 seed로 확정하지 않습니다.
5. Patch length와 mask ratio를 고정한 뒤 Ridge/CNN/Transformer estimator를 만들고
   Random/Easy/Medium/Hard 및 predictability-aware 비교로 넘어갑니다.
6. Full encoder가 600 tokens를 보는 linear probe feature extraction도 시간/메모리를 쓰므로 기록된 시간을 확인하세요.
   OOM이면 모든 조건에서 공통 batch를 줄여 다시 시작합니다. 한 조건만 batch/update를 바꾸지 마세요.
7. 서버에는 artifacts 전체와 이 notebook을 옮기고 경로를 지정하세요. GPU 노드 할당 후
   torch.cuda.is_available()가 True인지 확인해야 합니다. 로그인 노드에서 학습하지 마세요.


In [ ]:
if not RESULTS.empty:
    display(RESULTS.sort_values("macro_auroc", ascending=False))
    if MODE == "smoke":
        print("이 결과는 smoke test입니다. 최적 patch/mask ratio의 근거로 사용하지 마세요.")
    elif (RESULTS.valid_metric_classes != len(CLASSES)).any():
        print("모든 5개 클래스가 평가되지 않았습니다. subset을 점검하세요.")
    else:
        print("validation 탐색 결과입니다. 상위 2개 조건을 3개 seed/더 긴 학습으로 재검증하세요.")
else:
    print("현재 단계: patch/mask 검증 완료. 학습 결과는 아직 없습니다.")
